# LLM Colors Preference Analysis
Analyze the results of adaptive preference elicitation from LLMs on the colors domain, comparing GRUM to the Bradley-Terry baseline.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Set project root to import grums modules if needed
ROOT = Path.cwd().parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

sns.set_theme(style="whitegrid", palette="muted")

## 1. Load Data
Set the `EXP_DIR` to your experiment output folder.

In [ ]:
# TODO: SET your actual experiment directory here (relative to project root)
EXP_DIR = ROOT / "results" / "llm" / "llm_colors-20260325-185043"
output_dir = EXP_DIR / "outputs"

results = []
if output_dir.exists():
    for f in sorted(output_dir.glob("*.json")):
        with open(f, "r") as j:
            results.append(json.load(j))

if results:
    print(f"Loaded {len(results)} trials from {EXP_DIR.name} ({results[0]['model_id']})")
else:
    print("No results found! Set EXP_DIR to a valid experiment path.")

## 2. Global Preferences: GRUM vs Bradley-Terry
How does the learned intrinsic preference ($\delta$) compare to the social average (BT $\beta$)? 
We expect high correlation if prompt variations are indeed variations around a stable core.

In [ ]:
color_names = ["blue", "red", "green", "purple", "yellow"]

grum_deltas = []
bt_deltas = []

for r in results:
    history = r["history"]
    last_step = str(sorted(map(int, history.keys()))[-1])
    entry = history[last_step]
    
    if "grum" in entry:
        grum_deltas.append(entry["grum"]["delta"])
        bt_deltas.append(entry["bt"]["delta"])
    else:
        grum_deltas.append(entry["delta"])

if grum_deltas:
    df_grum = pd.DataFrame(grum_deltas, columns=color_names).melt(var_name="Color", value_name="Score")
    order = df_grum.groupby("Color")["Score"].mean().sort_values(ascending=False).index
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
    sns.barplot(data=df_grum, x="Color", y="Score", order=order, ax=axes[0])
    axes[0].set_title("GRUM Intrinsic Preferences ($\delta$)")
    
    if bt_deltas:
        df_bt = pd.DataFrame(bt_deltas, columns=color_names).melt(var_name="Color", value_name="Score")
        sns.barplot(data=df_bt, x="Color", y="Score", order=order, ax=axes[1])
        axes[1].set_title("Bradley-Terry Baseline ($\beta$)")
    
    plt.show()

### 2.1 Correlation Analysis

In [ ]:
if grum_deltas and bt_deltas:
    mean_grum = np.mean(grum_deltas, axis=0)
    mean_bt = np.mean(bt_deltas, axis=0)
    
    plt.figure(figsize=(8, 8))
    sns.regplot(x=mean_bt, y=mean_grum, scatter_kws={'s': 100})
    for i, name in enumerate(color_names):
        plt.annotate(name, (mean_bt[i], mean_grum[i]), xytext=(5, 5), textcoords='offset points')
    
    corr = np.corrcoef(mean_bt, mean_grum)[0, 1]
    plt.title(f"Intrinsic Preference Correlation: GRUM vs BT (R={corr:.3f})")
    plt.xlabel("Bradley-Terry Score")
    plt.ylabel("GRUM Delta Score")
    plt.show()

## 3. Persona Effect: Rank Reversals
Does the agent hidden state significantly alter preference ordering?

In [ ]:
if results and "grum" in results[0]["history"]["1"]:
    res = results[0]
    last_step = str(sorted(map(int, res["history"].keys()))[-1])
    delta = np.array(res["history"][last_step]["grum"]["delta"])
    B = np.array(res["history"][last_step]["grum"]["interaction"])
    
    # Sample two hypothetical agents at opposite ends of the PCA space
    K, M = B.shape
    x0_pos = np.zeros(K); x0_pos[0] = 2.0
    x0_neg = np.zeros(K); x0_neg[0] = -2.0
    
    u_pos = delta + x0_pos @ B
    u_neg = delta + x0_neg @ B
    
    df_rv = pd.DataFrame({
        "Color": color_names,
        "PCA0_High": u_pos,
        "PCA0_Low": u_neg
    }).melt(id_vars="Color", var_name="Persona", value_name="Utility")
    
    plt.figure(figsize=(12, 6))
    sns.lineplot(data=df_rv, x="Color", y="Utility", hue="Persona", marker="o")
    plt.title("Persona Effect: Predicted Preferences for Opposite PCA States")
    plt.axhline(0, color='black', alpha=0.2)
    plt.show()

## 4. Interaction Matrix ($B$)
Heatmap of how each PCA dimension modulates specific color preferences.

In [ ]:
all_B = []
for r in results:
    history = r["history"]
    ls = str(sorted(map(int, history.keys()))[-1])
    ent = history[ls]
    B = np.array(ent["grum"]["interaction"] if "grum" in ent else ent["interaction"])
    all_B.append(B)

if all_B:
    mean_B = np.mean(all_B, axis=0)
    plt.figure(figsize=(12, 8))
    sns.heatmap(mean_B, annot=True, cmap="coolwarm", center=0, 
                xticklabels=color_names, yticklabels=[f"PCA_{i}" for i in range(mean_B.shape[0])])
    plt.title("Mean Interaction Matrix (B)")
    plt.show()

## 5. Convergence Stability
Tracking parameter stabilization across rounds.

In [ ]:
if results:
    fig, ax = plt.subplots(figsize=(12, 6))
    for i, r in enumerate(results[:3]):
        steps = sorted(map(int, r["history"].keys()))
        for j, c in enumerate(color_names):
            y = [r["history"][str(s)]["grum"]["delta"][j] if "grum" in r["history"][str(s)] else r["history"][str(s)]["delta"][j] for s in steps]
            alpha = 1.0 if i == 0 else 0.3
            ax.plot(steps, y, label=c if i == 0 else "", color=sns.color_palette()[j], alpha=alpha)
    
    plt.title("Stability of Delta Parameters across Rounds (3 Trials)")
    plt.xlabel("Observation Round")
    plt.ylabel("Estimated Delta")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.show()